# Face Detection + Emotion Classification Pipeline
Two-stage pipeline:
1. **`yolov8n-face.pt`** — detects faces (bbox + detection confidence)
2. **Your pretrained emotion model** — classifies cropped face (7–9 emotion classes)

**Supports:** single image, multiple images, folder (zip), zip of images, video, and a pseudo-live webcam feed (browser snapshot loop).

**Before running:** upload `yolov8n-face.pt` and your emotion model weights (e.g. `emotion_model.pt`) when prompted in Cell 3, or place them in Google Drive and update the paths in Cell 4.


## 1. Install dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless pillow pandas tqdm


## 2. Imports

In [ ]:
import os, io, shutil, zipfile, uuid, time, base64
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

from ultralytics import YOLO

import matplotlib.pyplot as plt
from IPython.display import display, Javascript, Image as IPyImage, HTML, clear_output
from google.colab import files as colab_files
from google.colab.output import eval_js

print("Imports OK")


## 3. Upload model weights
Run this cell and upload:
- `yolov8n-face.pt` (face detector)
- your emotion classification weights (e.g. `emotion_model.pt`)

Alternatively, skip this cell and set `FACE_MODEL_PATH` / `EMOTION_MODEL_PATH` in Cell 4 to a Google-Drive path (mount Drive first with `from google.colab import drive; drive.mount('/content/drive')`).


In [ ]:
WORK_DIR = "/content/emotion_pipeline"
MODEL_DIR = f"{WORK_DIR}/models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("Upload yolov8n-face.pt and your emotion model .pt file (multi-select allowed):")
uploaded = colab_files.upload()
for fname, content in uploaded.items():
    dest = os.path.join(MODEL_DIR, fname)
    with open(dest, "wb") as f:
        f.write(content)
    print(f"Saved -> {dest}")


## 4. Config

In [ ]:
FACE_MODEL_PATH    = f"{MODEL_DIR}/yolov8n-face.pt"      # update filename if different
EMOTION_MODEL_PATH  = f"{MODEL_DIR}/emotion_model.pt"      # update to your actual filename

# Set True if your emotion model is a YOLO **classification** model (yolov8*-cls based).
# Set False if it's a plain torch/timm model with a custom head (you'll need to adapt
# `EmotionClassifier.predict` in Cell 9 accordingly).
EMOTION_MODEL_IS_YOLO_CLS = True

FACE_CONF_THRESH     = 0.40     # min confidence to accept a face detection
EMOTION_CONF_THRESH  = 0.00     # min confidence to accept an emotion label (0 = always keep top1)
FACE_CROP_MARGIN     = 0.15     # expand bbox by this fraction on each side before cropping
ALLOWED_EXTENSIONS   = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VIDEO_EXTENSIONS     = {".mp4", ".avi", ".mov", ".mkv"}
MIN_RESOLUTION       = 20       # reject images smaller than this (px) on either side
VIDEO_FRAME_STRIDE   = 5        # process every Nth frame (speed/accuracy tradeoff)

OUTPUT_DIR = f"{WORK_DIR}/outputs"
EXTRACT_DIR = f"{WORK_DIR}/extracted"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)

print("Config set. Face model:", FACE_MODEL_PATH, "| Emotion model:", EMOTION_MODEL_PATH)


## 5. Validation utilities
Checks each image: not corrupted, allowed format, minimum resolution. Invalid files are
skipped (not failed) and logged in a rejection report.


In [ ]:
class ValidationResult:
    def __init__(self, is_valid, reason=None):
        self.is_valid = is_valid
        self.reason = reason

def validate_image_file(path):
    """Validate a single image file on disk. Returns ValidationResult."""
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return ValidationResult(False, "File missing or empty")

    if p.suffix.lower() not in ALLOWED_EXTENSIONS:
        return ValidationResult(False, f"Unsupported format '{p.suffix}'")

    try:
        with Image.open(p) as img:
            img.verify()  # checks integrity, but closes the file afterwards
    except (UnidentifiedImageError, OSError, SyntaxError) as e:
        return ValidationResult(False, f"Corrupted/unreadable image: {e}")

    try:
        with Image.open(p) as img:
            w, h = img.size
            if w < MIN_RESOLUTION or h < MIN_RESOLUTION:
                return ValidationResult(False, f"Resolution too small ({w}x{h})")
    except Exception as e:
        return ValidationResult(False, f"Could not read dimensions: {e}")

    return ValidationResult(True)


def validate_video_file(path):
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return ValidationResult(False, "File missing or empty")
    if p.suffix.lower() not in VIDEO_EXTENSIONS:
        return ValidationResult(False, f"Unsupported video format '{p.suffix}'")
    cap = cv2.VideoCapture(str(p))
    ok = cap.isOpened()
    if ok:
        ret, _ = cap.read()
        ok = ret
    cap.release()
    if not ok:
        return ValidationResult(False, "Video could not be opened/decoded")
    return ValidationResult(True)

print("Validation utilities ready")


## 6. Input collectors
Normalizes every input type (single image / multiple images / zip of images / zip of a
folder / video) into one uniform list of `{path, source}` records, plus a separate
rejection log for invalid files.


In [ ]:
def fresh_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)
    return path


def collect_from_upload(save_dir):
    """Prompts a browser upload dialog. Handles: single image, multiple images,
    a .zip of images, or a .zip of a folder (nested). Returns (image_paths, rejections)."""
    fresh_dir(save_dir)
    print("Select file(s) to upload (images and/or a .zip):")
    uploaded = colab_files.upload()

    image_paths, rejections = [], []

    for fname, content in uploaded.items():
        raw_path = os.path.join(save_dir, fname)
        with open(raw_path, "wb") as f:
            f.write(content)

        if fname.lower().endswith(".zip"):
            extract_to = os.path.join(save_dir, f"extracted_{Path(fname).stem}")
            os.makedirs(extract_to, exist_ok=True)
            try:
                with zipfile.ZipFile(raw_path, "r") as z:
                    z.extractall(extract_to)
            except zipfile.BadZipFile:
                rejections.append({"file": fname, "reason": "Corrupted zip archive"})
                continue

            # walk recursively -> handles "zip of images" AND "zip of a folder"
            for root, _, files in os.walk(extract_to):
                for fn in files:
                    if fn.startswith("__MACOSX") or fn.startswith("."):
                        continue
                    full = os.path.join(root, fn)
                    res = validate_image_file(full)
                    if res.is_valid:
                        image_paths.append(full)
                    else:
                        rejections.append({"file": os.path.relpath(full, extract_to), "reason": res.reason})
        else:
            res = validate_image_file(raw_path)
            if res.is_valid:
                image_paths.append(raw_path)
            else:
                rejections.append({"file": fname, "reason": res.reason})

    print(f"Collected {len(image_paths)} valid image(s), {len(rejections)} rejected.")
    return image_paths, rejections


def collect_video_upload(save_dir):
    fresh_dir(save_dir)
    print("Select a video file to upload:")
    uploaded = colab_files.upload()
    for fname, content in uploaded.items():
        raw_path = os.path.join(save_dir, fname)
        with open(raw_path, "wb") as f:
            f.write(content)
        res = validate_video_file(raw_path)
        if res.is_valid:
            print(f"Video OK: {fname}")
            return raw_path
        else:
            print(f"Rejected '{fname}': {res.reason}")
    return None

print("Input collectors ready")


## 7. Load models

In [ ]:
assert os.path.exists(FACE_MODEL_PATH), f"Face model not found at {FACE_MODEL_PATH} (check Cell 3 upload / Cell 4 path)"
assert os.path.exists(EMOTION_MODEL_PATH), f"Emotion model not found at {EMOTION_MODEL_PATH} (check Cell 3 upload / Cell 4 path)"

face_model = YOLO(FACE_MODEL_PATH)
emotion_model = YOLO(EMOTION_MODEL_PATH)  # assumes YOLO-cls; see EMOTION_MODEL_IS_YOLO_CLS

EMOTION_CLASS_NAMES = emotion_model.names
print("Emotion classes:", EMOTION_CLASS_NAMES)
print("Models loaded.")


## 8. Face detector wrapper

In [ ]:
def expand_bbox(x1, y1, x2, y2, w, h, margin=FACE_CROP_MARGIN):
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, x1 - bw * margin)
    y1 = max(0, y1 - bh * margin)
    x2 = min(w, x2 + bw * margin)
    y2 = min(h, y2 + bh * margin)
    return int(x1), int(y1), int(x2), int(y2)


def detect_faces(image_bgr, conf_thresh=FACE_CONF_THRESH):
    """Returns list of dicts: bbox (x1,y1,x2,y2), det_conf."""
    h, w = image_bgr.shape[:2]
    results = face_model.predict(image_bgr, conf=conf_thresh, verbose=False)
    faces = []
    for r in results:
        if r.boxes is None:
            continue
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            x1, y1, x2, y2 = expand_bbox(x1, y1, x2, y2, w, h)
            faces.append({
                "bbox": (x1, y1, x2, y2),
                "det_conf": float(box.conf[0]),
            })
    return faces

print("Face detector ready")


## 9. Emotion classifier wrapper

In [ ]:
def classify_emotion(face_crop_bgr):
    """Returns dict: label, cls_conf, all_scores (dict class->prob).
    Adapt this function if EMOTION_MODEL_IS_YOLO_CLS is False (custom torch model)."""
    if face_crop_bgr.size == 0:
        return {"label": "N/A", "cls_conf": 0.0, "all_scores": {}}

    if EMOTION_MODEL_IS_YOLO_CLS:
        results = emotion_model.predict(face_crop_bgr, verbose=False)
        probs = results[0].probs
        top1 = int(probs.top1)
        label = EMOTION_CLASS_NAMES[top1]
        conf = float(probs.top1conf)
        all_scores = {EMOTION_CLASS_NAMES[i]: float(p) for i, p in enumerate(probs.data.tolist())}
        return {"label": label, "cls_conf": conf, "all_scores": all_scores}
    else:
        # --- Plug in your custom (non-YOLO) inference here, e.g. torch/timm model ---
        # img = preprocess(face_crop_bgr)
        # logits = custom_model(img)
        # probs = softmax(logits)
        # ...
        raise NotImplementedError("Add custom emotion-model inference here")

print("Emotion classifier ready")


## 10. Drawing utilities

In [ ]:
def draw_annotations(image_bgr, faces_with_emotions):
    img = image_bgr.copy()
    for f in faces_with_emotions:
        x1, y1, x2, y2 = f["bbox"]
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"{f['emotion_label']} {f['emotion_conf']*100:.1f}% | face {f['det_conf']*100:.1f}%"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(img, (x1, max(0, y1 - th - 8)), (x1 + tw + 4, y1), (0, 255, 0), -1)
        cv2.putText(img, label, (x1 + 2, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    return img


def show_bgr(image_bgr, title=None, figsize=(8, 8)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

print("Drawing utilities ready")


## 11. Core per-image pipeline
Runs steps 3–5 (detect -> crop -> classify) for a single image and returns per-face
records plus the annotated image.


In [ ]:
def process_single_image(image_path_or_array, image_name="image"):
    if isinstance(image_path_or_array, (str, Path)):
        img_bgr = cv2.imread(str(image_path_or_array))
        if img_bgr is None:
            return [], None, f"Could not decode image: {image_path_or_array}"
    else:
        img_bgr = image_path_or_array

    faces = detect_faces(img_bgr)
    records = []

    if not faces:
        records.append({
            "image_name": image_name, "face_id": None,
            "bbox": None, "face_conf": None,
            "emotion_label": "NO_FACE_DETECTED", "emotion_conf": None,
        })
        return records, img_bgr, None

    faces_with_emotions = []
    for i, f in enumerate(faces):
        x1, y1, x2, y2 = f["bbox"]
        crop = img_bgr[y1:y2, x1:x2]
        emo = classify_emotion(crop)
        rec = {
            "image_name": image_name,
            "face_id": i,
            "bbox": f["bbox"],
            "face_conf": round(f["det_conf"], 4),
            "emotion_label": emo["label"],
            "emotion_conf": round(emo["cls_conf"], 4),
        }
        records.append(rec)
        faces_with_emotions.append({**f, "emotion_label": emo["label"], "emotion_conf": emo["cls_conf"]})

    annotated = draw_annotations(img_bgr, faces_with_emotions)
    return records, annotated, None

print("Core pipeline ready")


## 12. Batch pipeline (images) -> table + CSV download
Works for single image, multiple images, and zip/folder uploads (same collected list).


In [ ]:
def run_batch_pipeline(image_paths, save_annotated=True):
    all_records = []
    annotated_dir = os.path.join(OUTPUT_DIR, "annotated_images")
    if save_annotated:
        fresh_dir(annotated_dir)

    for path in tqdm(image_paths, desc="Processing images"):
        name = os.path.basename(path)
        records, annotated, err = process_single_image(path, image_name=name)
        if err:
            all_records.append({"image_name": name, "face_id": None, "bbox": None,
                                 "face_conf": None, "emotion_label": f"ERROR: {err}", "emotion_conf": None})
            continue
        all_records.extend(records)
        if save_annotated and annotated is not None:
            cv2.imwrite(os.path.join(annotated_dir, name), annotated)

    df = pd.DataFrame(all_records)
    return df, annotated_dir


def export_csv_and_download(df, filename="predictions.csv"):
    csv_path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")
    colab_files.download(csv_path)
    return csv_path

print("Batch pipeline ready")


## 13. Demo — run on uploaded image(s) / zip / folder
Run the next cell, upload a single image, several images, or a `.zip` (of images or of a
folder) — all handled the same way.


In [ ]:
img_paths, rejections = collect_from_upload(os.path.join(EXTRACT_DIR, "batch_upload"))

if rejections:
    print("\nRejected files:")
    display(pd.DataFrame(rejections))

df_results, annotated_dir = run_batch_pipeline(img_paths)
display(df_results)

# Show first annotated image inline as a sanity check
if len(img_paths) > 0:
    first_name = os.path.basename(img_paths[0])
    first_annotated = os.path.join(annotated_dir, first_name)
    if os.path.exists(first_annotated):
        show_bgr(cv2.imread(first_annotated), title=first_name)

csv_path = export_csv_and_download(df_results, filename="emotion_predictions.csv")


## 14. Video pipeline
Processes a video frame-by-frame (every `VIDEO_FRAME_STRIDE` frames), runs the same
detect -> crop -> classify pipeline per frame, writes an annotated output video, and
produces a frame-level CSV.


In [ ]:
def process_video(video_path, frame_stride=VIDEO_FRAME_STRIDE, save_annotated_video=True):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_path = os.path.join(OUTPUT_DIR, "annotated_video.mp4")
    writer = None
    if save_annotated_video:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(out_path, fourcc, fps / frame_stride, (w, h))

    records = []
    frame_idx = 0
    pbar = tqdm(total=total_frames, desc="Processing video")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_stride == 0:
            faces = detect_faces(frame)
            faces_with_emotions = []
            if not faces:
                records.append({"frame": frame_idx, "timestamp_s": round(frame_idx / fps, 2),
                                 "face_id": None, "bbox": None, "face_conf": None,
                                 "emotion_label": "NO_FACE_DETECTED", "emotion_conf": None})
            for i, f in enumerate(faces):
                x1, y1, x2, y2 = f["bbox"]
                crop = frame[y1:y2, x1:x2]
                emo = classify_emotion(crop)
                records.append({
                    "frame": frame_idx, "timestamp_s": round(frame_idx / fps, 2),
                    "face_id": i, "bbox": f["bbox"], "face_conf": round(f["det_conf"], 4),
                    "emotion_label": emo["label"], "emotion_conf": round(emo["cls_conf"], 4),
                })
                faces_with_emotions.append({**f, "emotion_label": emo["label"], "emotion_conf": emo["cls_conf"]})

            if writer is not None:
                annotated = draw_annotations(frame, faces_with_emotions)
                writer.write(annotated)

        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    if writer is not None:
        writer.release()

    df = pd.DataFrame(records)
    return df, (out_path if save_annotated_video else None)

print("Video pipeline ready")


## 15. Demo — run on uploaded video

In [ ]:
video_path = collect_video_upload(os.path.join(EXTRACT_DIR, "video_upload"))

if video_path:
    df_video, annotated_video_path = process_video(video_path)
    display(df_video.head(20))
    csv_path = export_csv_and_download(df_video, filename="video_emotion_predictions.csv")
    if annotated_video_path and os.path.exists(annotated_video_path):
        print(f"Annotated video saved: {annotated_video_path}")
        colab_files.download(annotated_video_path)
else:
    print("No valid video uploaded.")


## 16. Live / pseudo-live webcam feed
Colab notebooks run in the browser and cannot open a local camera device directly, so a
true `cv2.VideoCapture(0)` won't work here. Instead this uses a JS snippet to grab
snapshots from your **browser's webcam**, run each snapshot through the pipeline, and
display the annotated result — looped to behave like a live feed.

Run Cell 17 once to define the capture function, then run Cell 18 to start the loop
(grant camera permission when your browser prompts you).


In [ ]:
# Standard Colab webcam JS helper (captures a single JPEG frame as base64)
def take_photo_js(quality=0.8):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      div.appendChild(video);
      document.body.appendChild(div);
      video.srcObject = stream;
      await video.play();

      // Give the camera a moment to adjust
      await new Promise((resolve) => setTimeout(resolve, 600));

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data_url = eval_js('takePhoto({})'.format(quality))
    header, encoded = data_url.split(",", 1)
    binary = base64.b64decode(encoded)
    arr = np.frombuffer(binary, dtype=np.uint8)
    frame_bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    return frame_bgr

print("Webcam capture function ready")


In [ ]:
def run_live_webcam_loop(num_captures=10, pause_seconds=1.0):
    """Pseudo-live feed: repeatedly grabs a browser webcam snapshot, runs the pipeline,
    and displays the annotated result + a running results table.
    Stop early with Runtime > Interrupt execution."""
    all_records = []
    for i in range(num_captures):
        clear_output(wait=True)
        print(f"Capture {i+1}/{num_captures} — click 'Allow' if prompted for camera access.")
        try:
            frame = take_photo_js()
        except Exception as e:
            print("Capture failed (camera permission denied or unavailable):", e)
            break

        name = f"live_frame_{i:03d}"
        records, annotated, err = process_single_image(frame, image_name=name)
        if err:
            print("Error:", err)
            continue
        all_records.extend(records)
        if annotated is not None:
            show_bgr(annotated, title=f"Live capture {i+1}")
        display(pd.DataFrame(all_records).tail(10))
        time.sleep(pause_seconds)

    df_live = pd.DataFrame(all_records)
    return df_live

# Example run:
# df_live = run_live_webcam_loop(num_captures=10, pause_seconds=1.0)
# export_csv_and_download(df_live, filename="live_feed_predictions.csv")


## Notes / things to double-check
- **`EMOTION_MODEL_IS_YOLO_CLS`**: if your emotion model isn't a YOLO classification export
  (e.g. it's a custom CNN/torch `.pt` state dict), implement the `else` branch in Cell 9's
  `classify_emotion()`.
- **Class names**: `EMOTION_CLASS_NAMES` is pulled from the model itself (`emotion_model.names`);
  verify it matches your 7–9 expected labels after loading (Cell 7 output).
- **Face-crop margin**: `FACE_CROP_MARGIN` (Cell 4) expands the bbox before cropping — tune
  this if your classifier was trained on tighter/looser crops.
- **Thresholds**: `FACE_CONF_THRESH` and `EMOTION_CONF_THRESH` are exposed in Cell 4 for
  quick tuning without touching pipeline code.
- **Folder uploads**: browsers can't upload a folder directly in Colab's file picker — zip
  the folder first, the collector (Cell 6) walks it recursively either way.
- **GPU**: Runtime > Change runtime type > GPU, for faster inference on video/batches.
- **True real-time streaming** (RTSP/webcam at full framerate) is not practical inside
  Colab's browser sandbox — for that, run this same pipeline code as a local Python script
  with `cv2.VideoCapture(0)`, or deploy with `streamlit-webrtc` outside Colab.
